<a href="https://colab.research.google.com/github/lisa6125/colab/blob/main/%E3%80%8C2026_AIML_LSTM_Stock_Prediction_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — LSTM Stock Price Prediction

使用台達電（股票代號 2308）的歷史股價，透過 LSTM 預測下一個**交易日**的收盤價（close）。


## 作業：比較不同 time frame（lookback window）的預測效果

### 目標
請保持 **LSTM 架構、特徵欄位、資料切分比例、random seed、epoch、learning rate** 等設定一致，
請維持相同的 LSTM 模型、feature、target、資料切分與訓練設定，只修改 WINDOW。分別完成 WINDOW = 5、10、20 三組實驗，每組重新訓練模型。

- `WINDOW = 5`：使用前 **5 個交易日**的特徵，預測**第 6 個交易日**的 close。
- `WINDOW = 10`：使用前 **10 個交易日**，預測第 11 個交易日的 close。
- `WINDOW = 20`：使用前 **20 個交易日**，預測第 21 個交易日的 close。

**開始前請先在 Colab 選「檔案 → 在雲端硬碟中儲存副本」**

### 如何操作（每一組都要訓練自己的模型）
1. 第一次從上到下執行所有儲存格，記錄 `WINDOW = 5` 的 **Validation MSE** 與預測圖。
2. 到 **Dataset** 區塊把 `WINDOW` 改成 10；**從 Dataset 到 Evaluation 的所有儲存格重新執行**，記錄結果。
3. 再改成 20，重複上述步驟。
4. 在報告比較三種設定的 Validation MSE 與預測曲線，說明你的觀察與可能原因。

> 本例採用 one-step-ahead prediction，每次預測下一個交易日的 close。WINDOW 代表過去輸入資料的交易日數。

### 評估指標
- **Validation MSE**（主要指標；還原為原始收盤價單位後計算，越小越好）
- Validation MAE（輔助參考）

| WINDOW（交易日） | Validation MSE | Validation MAE（選填） | 觀察 |
|---|---|---|---|
| 5 |  |  |  |
| 10 |  |  |  |
| 20 |  |  |  |

報告必須包含 WINDOW = 5、10、20 的 Validation MSE 比較表、三張 Validation Prediction 圖，以及對實驗結果的分析與結論。


## Download Data and Import Libraries

這份 CSV 包含 2020/01/02～2024/12/31 的台達電（2308）交易資料。程式會從 Google Drive 下載 `2308.csv`


In [ ]:
%pip install -q gdown

In [ ]:
from pathlib import Path

csv_path_2308 = Path('./2308.csv')

if not csv_path_2308.is_file():
    import gdown  # Installed in the dependency cell above
    file_id = '1cRaFFJNaQYUewKHXoULhUTveuihOx6tU'
    try:
        gdown.download(id=file_id, output=str(csv_path_2308), quiet=False)
    except Exception as exc:
        raise RuntimeError(
            'CSV 下載失敗。'
        ) from exc

if not csv_path_2308.is_file() or csv_path_2308.stat().st_size == 0:
    raise FileNotFoundError(
        '找不到 2308.csv。'
    )

print('Dataset:', csv_path_2308.resolve())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


In [ ]:
df_2308 = pd.read_csv(csv_path_2308)
# CSV 日期如 2020/1/2，必須先轉成 datetime 再排序（不能直接排序字串）。
df_2308['date'] = pd.to_datetime(df_2308['date'])
df_2308 = df_2308.sort_values('date').reset_index(drop=True)

assert df_2308['date'].is_monotonic_increasing, '日期沒有按時間排序'
print('Records:', len(df_2308))
print('Date range:', df_2308['date'].min().date(), '~', df_2308['date'].max().date())
df_2308.head(10)


### Train/Valid Split and Feature Normalization

We use the **first 85% of trading days** for training and the **last 15%** for validation.
The validation period starts at **2024-04-09** for the provided CSV (you can check the printed pivot date below).
The dataset is split **chronologically**, not randomly.

Because `open`, `close`, `low`, `high` and `volume` have different scales, we use `StandardScaler`:

$$x_{scaled} = (x - mean_{train}) / std_{train}$$

**Important:** The scaler is fitted on **training data only**. Validation statistics must not be used
for normalization because they would leak future information into training.
Standardization can improve training stability, though it does not eliminate all exploding/vanishing gradient problems.

To convert standardized values back to their original unit:

```python
raw_open = normalized_open * scaler.scale_[0] + scaler.mean_[0]
raw_close = normalized_close * scaler.scale_[1] + scaler.mean_[1]
raw_low = normalized_low * scaler.scale_[2] + scaler.mean_[2]
raw_high = normalized_high * scaler.scale_[3] + scaler.mean_[3]
raw_volume = normalized_volume * scaler.scale_[4] + scaler.mean_[4]
```

The reported **Validation MSE / MAE are calculated after the close prices are transformed back to the original unit**. The training loss plot, in contrast, uses standardized close values.


In [ ]:
pivot = round(0.85 * len(df_2308))
print('Validation starts:', df_2308.iloc[pivot]['date'].date())

df_train = df_2308.iloc[:pivot].copy()
df_valid = df_2308.iloc[pivot:].copy()

FEATURES_STD = ['open', 'close', 'low', 'high', 'volume']
scaler = StandardScaler().fit(df_train[FEATURES_STD])  # Fit on TRAIN only

df_train[FEATURES_STD] = scaler.transform(df_train[FEATURES_STD])
df_valid[FEATURES_STD] = scaler.transform(df_valid[FEATURES_STD])

print(f'#train days: {len(df_train)}, #valid days: {len(df_valid)}')
df_train.head(10)


# Dataset — Sliding Window

For example, if `WINDOW = 10`, each sample contains:

1. **Feature**: the previous **10 trading days** of `open`, `close`, `low`, `high`, `volume`, shape **[10, 5]**.
2. **Target**: the **next trading day's** `close`, shape **[1]**.

The label is always one trading day after the input window (not at its last day).

To make the window comparison fair, each experiment evaluates **the same validation target dates**.
For the first validation target, we use the last `WINDOW` training days as its past input;
we do **not** use any future validation target as an input. Later validation predictions may use
previously observed validation days, which is appropriate for one-step-ahead evaluation.


In [ ]:
class CompanyStockData(Dataset):
    def __init__(self, dataframe, window=10):
        super().__init__()
        if window < 1 or len(dataframe) <= window:
            raise ValueError('WINDOW 必須至少 1，且小於輸入資料筆數。')
        self.df = dataframe.reset_index(drop=True)
        self.window = window

    def __len__(self):
        # N 個交易日可以組成 N - WINDOW 個「過去 WINDOW 天 -> 下一天」樣本。
        return len(self.df) - self.window

    def __getitem__(self, idx):
        # Input: rows idx ... idx + WINDOW - 1 (past WINDOW days).
        feature = self.df.iloc[idx:idx + self.window][FEATURES_STD].to_numpy(dtype=np.float32)
        feature = torch.from_numpy(feature)  # [WINDOW, 5]

        # Target: row idx + WINDOW (the following trading day).
        target = float(self.df.iloc[idx + self.window]['close'])
        target = torch.tensor([target], dtype=torch.float32)  # [1]
        return feature, target


# ===== 學生實驗時，只改這裡的 WINDOW（例如 5 / 10 / 20） =====
WINDOW = 5

train_set = CompanyStockData(df_train, window=WINDOW)

# Include preceding training days as history, so every WINDOW tests the same
# validation target dates; no validation target is included in training.
valid_df = pd.concat([df_train.tail(WINDOW), df_valid], ignore_index=True)
valid_set = CompanyStockData(valid_df, window=WINDOW)

print(f'WINDOW={WINDOW} (use past {WINDOW} days to predict the next day)')
print('#train samples:', len(train_set))
print('#valid samples:', len(valid_set))
assert len(valid_set) == len(df_valid), 'Validation sample count should not depend on WINDOW.'
feature, target = valid_set[0]
print('One sample feature shape:', tuple(feature.shape))  # (WINDOW, 5)
print('One sample target shape:', tuple(target.shape))    # (1,)


# Model

We use the **same single-direction LSTM** in all three experiments. The last time step's output
is sent to a linear layer to predict next-day close.

Other models below are optional examples, **not required** for the WINDOW comparison.


In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1, num_classes=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True
        )
        self.regressor = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x, (h, c) = self.lstm(x)
        x = self.regressor(x[:, -1, :])  # Last time step -> next-day close
        return x


train_loader = DataLoader(train_set, shuffle=True, batch_size=8, num_workers=0)
valid_loader = DataLoader(valid_set, shuffle=False, batch_size=8, num_workers=0)

feature_b, target_b = next(iter(valid_loader))
print('Feature batch shape:', tuple(feature_b.shape))  # [N, WINDOW, 5]
print('Target batch shape:', tuple(target_b.shape))    # [N, 1]

# A quick shape check. A fresh model will be initialized before training.
model_example = LSTM(input_size=5, hidden_size=32, num_classes=1)
output_b = model_example(feature_b)
print('Prediction batch shape:', tuple(output_b.shape))  # [N, 1]


Some other RNN models you may explore **after** completing the required LSTM window comparison:
BiLSTM and GRU. Keep the model choice fixed when comparing `WINDOW` values.


In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BiLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])
        return x


class GRU(nn.Module):
    def __init__(self, input_size, hidden_size=1, num_layers=1, num_classes=1):
        super().__init__()

        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x, _ = self.gru(x)
        x = self.fc(x[:, -1, :])
        return x

# Training

Each time you change `WINDOW`, rerun **Dataset -> Model -> Training -> Evaluation** so that
new DataLoaders are created and the LSTM is trained **from scratch**.

For comparability, we use the same random seed, model architecture, 20 epochs, batch size,
optimizer and learning rate. Some GPU operations may still show small nondeterministic differences.

`Train loss` and `Valid loss` below are MSE on the **standardized** close values;
`Validation MSE` in the evaluation section is on the **original** price scale.


In [ ]:
# Fix seed for more comparable experiments (same architecture and training settings).
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = LSTM(input_size=5, hidden_size=32, num_layers=1, num_classes=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 20

history = {'train_loss': [], 'valid_loss': []}

for epoch in tqdm(range(EPOCHS), desc=f'Training WINDOW={WINDOW}'):
    # Train
    model.train()
    train_loss_sum = 0.0
    for feature_b, target_b in train_loader:
        feature_b = feature_b.to(device)
        target_b = target_b.to(device)

        optimizer.zero_grad()
        predict_b = model(feature_b)
        loss = criterion(predict_b, target_b)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * feature_b.size(0)
    train_loss = train_loss_sum / len(train_set)

    # Validation: no gradient computation or parameter update.
    model.eval()
    valid_loss_sum = 0.0
    with torch.no_grad():
        for feature_b, target_b in valid_loader:
            feature_b = feature_b.to(device)
            target_b = target_b.to(device)
            predict_b = model(feature_b)
            loss = criterion(predict_b, target_b)
            valid_loss_sum += loss.item() * feature_b.size(0)
    valid_loss = valid_loss_sum / len(valid_set)

    history['train_loss'].append(train_loss)
    history['valid_loss'].append(valid_loss)
    print(f'Epoch {epoch+1:02d}/{EPOCHS}: Train={train_loss:.6f}, Valid={valid_loss:.6f}')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train')
ax.plot(range(1, EPOCHS + 1), history['valid_loss'], label='Validation')
ax.set(title=f'Loss (WINDOW={WINDOW}, standardized close)', xlabel='Epoch', ylabel='MSE')
ax.legend()
plt.tight_layout()
plt.show()


## Evaluation

To interpret the errors in the **original stock price unit**, we first invert
`StandardScaler` for the **close** column using the training mean and standard deviation.

**For your report, compare the Validation MSE printed below** (not the standardized training loss).
Every `WINDOW` evaluates exactly the same validation target dates: **2024-04-09 to 2024-12-31**.


In [ ]:
@torch.no_grad()
def evaluate(dataset):
    model.eval()
    preds = []
    trues = []
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    for feature_b, target_b in loader:
        predict_b = model(feature_b.to(device)).cpu()
        preds.append(predict_b.numpy())
        trues.append(target_b.numpy())
    return np.concatenate(preds), np.concatenate(trues)  # [N, 1] each


CLOSE_IDX = FEATURES_STD.index('close')


def plot_predictions(dataset, dates, split_name):
    # The predicted target is standardized close; invert only its scale.
    preds, trues = evaluate(dataset)
    preds = preds[:, 0] * scaler.scale_[CLOSE_IDX] + scaler.mean_[CLOSE_IDX]
    trues = trues[:, 0] * scaler.scale_[CLOSE_IDX] + scaler.mean_[CLOSE_IDX]

    mse = np.mean((preds - trues) ** 2)
    mae = np.mean(np.abs(preds - trues))

    assert len(dates) == len(trues), 'Date and prediction lengths do not match.'
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(dates, trues, label='True close')
    ax.plot(dates, preds, label='Predicted close')
    ax.set(title=f'{split_name} (WINDOW={WINDOW}) | MSE={mse:.3f}, MAE={mae:.3f}',
           xlabel='Date', ylabel='Close price (original units)')
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    return mse, mae


# Train target dates start AFTER the first WINDOW training days.
train_dates = df_train['date'].iloc[WINDOW:].to_numpy()
train_mse, train_mae = plot_predictions(train_set, train_dates, 'Train')

# Validation target dates are the SAME for WINDOW=5, 10, 20.
valid_dates = df_valid['date'].to_numpy()
valid_mse, valid_mae = plot_predictions(valid_set, valid_dates, 'Validation')

print(f'RESULT | WINDOW={WINDOW} | Validation MSE={valid_mse:.3f} | Validation MAE={valid_mae:.3f}')
print('請將此組 Validation MSE / MAE 與預測圖記錄到報告，再修改 WINDOW 並重新訓練。')
